In [ ]:
import pandas as pd
import re
import unicodedata
import html
import os
import numpy as np
from datetime import datetime

## Filter Configuration

In [ ]:
RELEVANT_KEYWORDS = [
    "dpr", "dprd", "anggota dpr", "gedung dpr", "bubarkan",
    "tunjangan", "gaji pejabat", "dpr joget",
    "demo", "demonstrasi", "unjuk rasa", "protes", "aksi",
    "massa aksi", "pendemo", "revolusi dimulai", "peringatan darurat",
    "17+8", "tuntutan rakyat",
    "affan", "ojol", "rantis", "dilindas", "justice for affan",
    "gas air mata", "brimob", "aparat", "polisi brutal",
    "water cannon", "korban demo", "mahasiswa ditangkap",
    "penjarahan", "kerusuhan", "rusuh",
    "sahroni", "nicholas saputra", "mundur pak",
    "ruu perampasan aset", "warga jaga warga",
    "bubarkandpr", "desakprabowo", "justiceforaffan",
    "polisipembunuhrakyat", "reformasipolri", "selamatkanindonesia",
    "orangtololsedunia", "demo25agustus", "demo28agustus",
]

SPAM_KEYWORDS = [
    "shopee", "voucher", "diskon", "promo", "gratis ongkir",
    "cashback", "affiliate", "link di bio", "cek keranjang",
    "spaylater", "tokopedia", "lazada", "jastip", "open bo",
    "gacor", "slot", "togel", "giveaway", "dana kaget", "saweria",
    "prediksi bola", "live score", "streaming gratis",
    "demo masak", "demo produk", "demo game", "demo aplikasi",
]

## 1. Extract: Take and Read Data

In [ ]:
def extract_data(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File {file_path} not found")

    df = pd.read_csv(
        file_path,
        delimiter=",",
        encoding='utf-8',
        on_bad_lines='skip',
        dtype={
            'conversation_id_str': str,
            'id_str': str,
            'user_id_str': str
        }
    )

    print(f"Total raw data: {df.shape[0]} lines, {df.shape[1]} columns")
    display(df.head())
    return df

## 2. Transform: Data Processing

### Filter Date Range

In [ ]:
def filter_date_range(df):

    TWITTER_DT_FORMAT = "%a %b %d %H:%M:%S %z %Y"

    df['created_at'] = pd.to_datetime(
        df['created_at'],
        format=TWITTER_DT_FORMAT,
        errors='coerce'
    )

    if df['created_at'].dt.tz is not None:
        df['created_at'] = df['created_at'].dt.tz_convert(None)

    before = len(df)
    df = df.dropna(subset=['created_at'])
    print(f"  Date failed parse line (discarded): {before - len(df)}")

    start_date = pd.to_datetime('2025-08-15')
    end_date   = pd.to_datetime('2025-09-09 23:59:59')
    df_filtered = df[(df['created_at'] >= start_date) & (df['created_at'] <= end_date)].copy()

    print(f"  Data remaining after date filter: {len(df_filtered)} lines")
    return df_filtered

### Relevance & Spam Filter

In [ ]:
def filter_spam_and_relevance(df):
    spam_set     = set(SPAM_KEYWORDS)
    relevant_set = set(RELEVANT_KEYWORDS)

    def is_valid(row):
        text = str(row.get('full_text', '')).lower()
        lang = str(row.get('lang', '')).lower().strip()

        if lang != 'in':
            return False

        if any(sp in text for sp in spam_set):
            return False

        return any(kw in text for kw in relevant_set)

    mask = df.apply(is_valid, axis=1)
    df_filtered = df[mask].copy()
    print(f"  Data remaining after content filter: {len(df_filtered)} lines")
    return df_filtered

### Clean and Normalize

In [ ]:
def clean_and_normalize(df):
    drop_cols = ['location', 'username']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

    before = len(df)
    df = df.drop_duplicates(subset=['id_str'])
    print(f"  Duplicates removed: {before - len(df)} lines")

    print("  Missing values per column:")
    print(df.isnull().sum().to_string())

    str_cols = ['image_url', 'in_reply_to_screen_name']
    num_cols = ['favorite_count', 'retweet_count', 'quote_count', 'reply_count']
    for col in str_cols:
        if col in df.columns:
            df[col] = df[col].fillna('')
    for col in num_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

    df['full_text_raw'] = df['full_text'].astype(str)

    def normalize(text):
        if not isinstance(text, str):
            return ''
        text = unicodedata.normalize('NFC', text)
        text = html.unescape(text)
        text = re.sub(r'http\S+', '', text)
        text = re.sub(r'RT @\w+:\s*', '', text)
        text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
        text = ' '.join(text.split())
        return text.strip()

    df['full_text'] = df['full_text_raw'].apply(normalize)

    before = len(df)
    df = df[df['full_text'].str.len() > 5]
    print(f"  Text too short (discarded): {before - len(df)} lines")

    return df

### Thread Reconstruction

In [ ]:
def infer_parent_thread(df):
    df['parent_id_str'] = ''
    df['is_root_tweet'] = False

    if 'conversation_id_str' not in df.columns:
        print("Column conversation_id_str not found, skip thread reconstruction.")
        return df

    is_root = df['conversation_id_str'] == df['id_str']
    df.loc[is_root, 'is_root_tweet'] = True

    is_reply = ~is_root & df['conversation_id_str'].notna() & (df['conversation_id_str'] != '')
    df.loc[is_reply, 'parent_id_str'] = df.loc[is_reply, 'conversation_id_str']

    df['replied_to_username'] = ''
    has_reply_target = (
        df['in_reply_to_screen_name'].notna() &
        (df['in_reply_to_screen_name'].str.strip() != '')
    )
    df.loc[has_reply_target, 'replied_to_username'] = (
        df.loc[has_reply_target, 'in_reply_to_screen_name'].str.strip()
    )

    n_root   = is_root.sum()
    n_reply  = is_reply.sum()
    n_single = (~is_root & ~is_reply).sum()

    print(f"  Root tweet (parent = None)         : {n_root:,}")
    print(f"  Reply in thread (has parent)       : {n_reply:,}")
    print(f"  Tweet standalone (without thread)  : {n_single:,}")

    return df

### Features Extraction

In [ ]:
def feature_extraction(df):
    df['hashtags'] = df['full_text_raw'].apply(
        lambda x: ','.join(h for h in re.findall(r'#(\w+)', str(x)))
    )
    
    df['mentions'] = df['full_text_raw'].apply(
        lambda x: ','.join(m for m in re.findall(r'@(\w+)', str(x)))
    )
    
    def classify_tweet_type(row):
        if str(row.get('parent_id_str', '')).strip() != '':
            return 'reply'
        else:
            return 'original'

    df['tweet_type'] = df.apply(classify_tweet_type, axis=1)

    df['has_image'] = df['image_url'].astype(str).str.len() > 5

    df['log_fav'] = np.log1p(df['favorite_count'])
    df['log_rt']  = np.log1p(df['retweet_count'])
    df['engagement_score'] = (
        df['log_fav'] + df['log_rt'] + df['reply_count'] + df['quote_count']
    ) / 4

    df['created_at'] = df['created_at'].dt.strftime('%Y-%m-%dT%H:%M:%S')

    return df

### Transform Process

In [ ]:
def transform_process(df):
    df = filter_date_range(df)
    df = filter_spam_and_relevance(df)
    df = clean_and_normalize(df)
    df = infer_parent_thread(df)
    df = feature_extraction(df)
    return df

## 3. Load: Export to CSV File

In [ ]:
def load_data(df, output_path):
    export_cols = [
        'id_str', 'conversation_id_str', 'user_id_str',
        'full_text',
        'created_at',
        'favorite_count', 'retweet_count', 'reply_count', 'quote_count',
        'lang', 'tweet_url', 'image_url', 'has_image',
        'engagement_score', 'log_fav', 'log_rt',
        'hashtags', 'mentions',
        'tweet_type', 'is_root_tweet',
        'parent_id_str',
        'in_reply_to_screen_name', 'replied_to_username',
    ]

    existing_cols = [c for c in export_cols if c in df.columns]
    df_out = df[existing_cols].copy()

    if 'parent_id_str' in df_out.columns:
        df_out['parent_id_str'] = df_out['parent_id_str'].fillna('')

    print(f"[LOAD] saving {len(df_out):,} data bersih ke {output_path}...")
    df_out.to_csv(
        output_path,
        sep=';',
        index=False,
        float_format='%.4f',
        na_rep=''
    )

    if 'tweet_type' in df_out.columns:
        type_counts = df_out['tweet_type'].value_counts()
        print(f"\n  Distribusi tipe tweet:")
        for ttype, count in type_counts.items():
            print(f"    {ttype:<12}: {count:>8,} ({count/len(df_out)*100:.1f}%)")

    if 'parent_id_str' in df_out.columns:
        n_with_parent = (df_out['parent_id_str'] != '').sum()
        print(f"\n  Post with parents                   : {n_with_parent:,}")
        print(f"  Post without parent (root/standalone) : {len(df_out) - n_with_parent:,}")

    print(f"\n  process ETL completed: {output_path}")

## Main Execution

In [ ]:
if __name__ == "__main__":
    input_csv  = "" # change this to your input file path
    output_csv = "" # change this to your desired output file path

    try:
        df_raw   = extract_data(input_csv)
        df_clean = transform_process(df_raw)
        load_data(df_clean, output_csv)

        print(f"\nPost before preprocessing  : {len(df_raw):,}")
        print(f"Post after preprocessing     : {len(df_clean):,}")
        retained = len(df_clean) / len(df_raw) * 100 if len(df_raw) > 0 else 0
        print(f"Data retention               : {retained:.1f}%")

    except Exception as e:
        import traceback
        print(f"\n Error: {e}")
        traceback.print_exc()